# RAG su referti radiologici — walkthrough di validazione

Pipeline **offline** end-to-end: corpus sintetico → chunking → retrieval TF-IDF → generazione grounded con citazioni → anti-allucinazione.

⚠️ Il corpus è **SINTETICO** (referti generati artificialmente, NON dati reali di pazienti).

Riferimento: Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, NeurIPS 2020, arXiv:2005.11401.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[0] / "src"))

from rag_reports.corpus import load_sample_corpus
from rag_reports.rag import RagPipeline
from rag_reports.retriever import build_retriever
from rag_reports.evaluation import evaluate_retrieval, evaluate_abstention

docs = load_sample_corpus()
print(f"Documenti nel corpus sintetico: {len(docs)}")
print("Tutti synthetic=True:", all(d['synthetic'] for d in docs))

## 1. Retrieval TF-IDF

In [ ]:
retr = build_retriever(docs, dense=False)
for r in retr.search("È presente uno pneumotorace?", k=3):
    print(f"[{r.score:.3f}] {r.chunk.doc_id}: {r.chunk.text[:70]}...")

## 2. Pipeline RAG (backend estrattivo, offline)

In [ ]:
rag = RagPipeline.from_documents(docs, backend="extractive")
ans = rag.answer("Ci sono segni di polmonite al lobo inferiore destro?")
print("Astenuto:", ans.abstained)
print("Fonti:", ans.source_ids)
print()
print(ans.text)

## 3. Anti-allucinazione (query fuori-corpus)

In [ ]:
ans = rag.answer("Qual è la capitale della Francia?")
print("Astenuto:", ans.abstained)
print("Risposta:", ans.text)
assert ans.abstained, "dovrebbe astenersi su query fuori-corpus"

## 4. Metriche: hit-rate@k e astensione

In [ ]:
retr_eval = evaluate_retrieval(retr, k_values=(1, 3, 5))
print("Hit-rate@k:", {k: round(v, 3) for k, v in retr_eval.hit_rate_at_k.items()})

abst = evaluate_abstention(rag)
print(f"Astensioni corrette (fuori-corpus): {abst.n_correct_abstentions}/{abst.n_queries}")